# 07 — Validation

Confusion matrix, IoU, precision and recall of the modelled inundation against two
independent, real observed-extent products: Copernicus EMS EMSR927 (the official rapid
mapping activation for this event) and HDX's `hot_flood_npl` observed flood extent
(27 August 2026). **Reported whatever the numbers are.**

In [1]:
import sys
sys.path.insert(0, "..")
from pathlib import Path
from analysis.exposure.validation import load_cems_events, load_hdx_flood_extent, compare_to_reference

cems = load_cems_events()
hdx = load_hdx_flood_extent()
print(f"CEMS EMSR927: {len(cems)} observed-event polygons across 4 AOIs (01, 02, 03, 05)")
print(f"  classification: {cems['event_type'].value_counts().to_dict()}")
print(f"  total area: {cems.to_crs(32645).geometry.area.sum()/1e6:.2f} km2")
print(f"HDX hot_flood_npl flood extent: {len(hdx)} polygon, {hdx.geometry.area.sum()/1e6:.2f} km2")

CEMS EMSR927: 50 observed-event polygons across 4 AOIs (01, 02, 03, 05)
  classification: {'6-Mass Movement': 50}
  total area: 19.54 km2
HDX hot_flood_npl flood extent: 1 polygon, 31.71 km2


**A finding worth stating up front: CEMS classifies its observed extent as
`6-Mass Movement / Landslide`, not `flood`.** This is not our own labelling choice — it is
how Copernicus itself categorised the 26 August 2026 event's visible damage. It is a
second, independent confirmation of the same physical reality Phase 3's calibration
already found: this was fundamentally a mass-movement event with a water component, not a
pure flood, and any water-only hydraulic model should be expected to under-represent its
full extent.

## Validation against both references, at two scenario volumes

In [2]:
results = []
for slug, label in [
    ("reference_v1.0_d30_full", "1.0 Mm3 / 30 min"),
    ("v5.0_d360_full", "5.0 Mm3 / 360 min (largest in grid)"),
]:
    cog = Path(f"../dist/scenario_grid/{slug}_peak_rise.tif")
    for ref_name, ref in [("CEMS EMSR927", cems), ("HDX flood extent", hdx)]:
        cm = compare_to_reference(cog, ref)
        results.append({
            "scenario": label, "reference": ref_name,
            "precision": cm.precision, "recall": cm.recall,
            "IoU": cm.iou, "F1": cm.f1,
            "TP": cm.true_positive, "FP": cm.false_positive, "FN": cm.false_negative,
        })

import pandas as pd
df = pd.DataFrame(results)
df

,scenario,reference,precision,recall,IoU,F1,TP,FP,FN
0,1.0 Mm3 / 30 min,CEMS EMSR927,0.407643,0.145273,0.119951,0.214208,1280,1860,7531
1,1.0 Mm3 / 30 min,HDX flood extent,0.973885,0.194369,0.193361,0.324061,3058,82,12675
2,5.0 Mm3 / 360 min (largest in grid),CEMS EMSR927,0.410007,0.186925,0.147303,0.256782,1647,2370,7164
3,5.0 Mm3 / 360 min (largest in grid),HDX flood extent,0.964152,0.246170,0.243938,0.392203,3873,144,11860


## Honest reading

**Precision is consistently high (0.41–0.97), recall is consistently low (0.15–0.25).**
In plain terms: almost everywhere our model says "this got wet," the official record
agrees. But our model only captures 15–25% of the area the real event actually affected.

**This is the same finding as Phase 3's calibration, from a completely independent
comparison.** There, a water-only 1D router underestimated peak flow depth by roughly 83%
against geo-pera's reconstructed heights. Here, a water-only inundation footprint captures
only a fifth to a quarter of the officially mapped disturbed area. Two different
methods, two different independent ground-truth sources, the same conclusion:

**A shallow-water solver is not a two-phase debris flow.** `02-TECHNICAL-SPEC.md` already
names this limitation; this is the second, independent number behind it.

**Recall improves, modestly, with a larger scenario volume** (0.145→0.187 against CEMS,
0.194→0.246 against HDX, moving from 1.0 to 5.0 Mm³) — in the expected direction, but
nowhere close to closing the gap. Volume alone does not make a water model into a debris
model.

**Precision near 1.0 against HDX is worth being honest about too.** It does not mean the
model is highly accurate — it means the model's footprint is conservative and narrow, so
almost everything it does flag happens to fall inside a much larger true extent. A model
that flagged nothing would have undefined precision and zero recall; a model that flagged
everything would have poor precision and perfect recall. High precision with low recall is
the signature of *under-prediction*, not of correctness.